In [1]:
import math
import numpy as np

# --- Hidden state mappings ---
hidden_states = dict(zip(["s", "E", "5", "I", "e"], range(5)))
state_labels  = dict(zip(range(5), ["s", "E", "5", "I", "e"]))

# --- Transition probability matrix (row = from, col = to) ---
trans_matrix = np.zeros((5, 5))
trans_matrix[0, 1] = 1.0
trans_matrix[1, 1] = 0.9
trans_matrix[1, 2] = 0.1
trans_matrix[2, 3] = 1.0
trans_matrix[3, 3] = 0.9
trans_matrix[3, 4] = 0.1

# --- Emission probability matrix (row = state, col = nucleotide) ---
nuc_index = {nuc: idx for idx, nuc in enumerate("ACGT")}

emit_matrix = np.zeros((5, 4))
emit_matrix[1] = [0.25, 0.25, 0.25, 0.25]  # Exon
emit_matrix[2] = [0.05, 0.00, 0.95, 0.00]  # 5'ss
emit_matrix[3] = [0.40, 0.10, 0.10, 0.40]  # Intron

# --- Observed sequence ---
observed_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [2]:
def compute_log_probability(path_str, obs_seq):
    """Return the log-probability of a given state path emitting obs_seq."""
    log_p = math.log(0.25)
    idx = 1
    while idx < len(path_str):
        src = hidden_states[path_str[idx - 1]]
        dst = hidden_states[path_str[idx]]
        combined = trans_matrix[src][dst] * emit_matrix[dst][nuc_index[obs_seq[idx]]]
        log_p += math.log(combined)
        idx += 1
    return log_p

In [3]:
# Path 1 : transition at position 7
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
path_a = "EEEEEE5IIIIIIIIIIIIIIIIIII"
k1 = compute_log_probability(path_a, observed_seq) + math.log(0.1)
print(k1)

-43.89740030179307


In [ ]:
# Path 2 : transition at position 9
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
path_b = "EEEEEEEE5IIIIIIIIIIIIIIIII"
k2 = compute_log_probability(path_b, observed_seq) + math.log(0.1)
print(k2)

-43.45111319916465


In [5]:
# Path 3 : transition at position 13
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
path_c = "EEEEEEEEEEEE5IIIIIIIIIIIII"
k3 = compute_log_probability(path_c, observed_seq) + math.log(0.1)
print(k3)

-43.944833355027704


In [6]:
# Path 4 : transition at position 16
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
path_d = "EEEEEEEEEEEEEEE5IIIIIIIIII"
k4 = compute_log_probability(path_d, observed_seq) + math.log(0.1)
print(k4)

-42.58225552052512


In [7]:
# Path 5 : transition at position 19
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
path_e = "EEEEEEEEEEEEEEEEEE5IIIIIII"
k5 = compute_log_probability(path_e, observed_seq) + math.log(0.1)
print(k5)

-41.21967768602254


In [8]:
# Path 6 : transition at position 23
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
path_f = "EEEEEEEEEEEEEEEEEEEEEE5III"
k6 = compute_log_probability(path_f, observed_seq) + math.log(0.1)
print(k6)

-41.713397841885595


In [9]:
# All-Exon path (no splice)
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
path_all_exon = "E" * len(observed_seq)
only_E = compute_log_probability(path_all_exon, observed_seq) + math.log(0.1)
print(only_E)

-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [10]:
# Initiate two matrices: 
# viterbi_value_matrix: to store the values described in the documentation above 
# viterbi_trace_matrix: to store the path the lead to the the maximum value in each cell
# For example, the first column of viterbi_trace_matrix will be 
# [0] indicating start state released `C`: even though not possible - but we just initiate
# [1] indicating Exon state released `C`:
# [2] indicating 5'ss state released `C`: even though not possible - but we just initiate
# [3] indicating Intron state released `C`: 
# [4] indicating end state released `C`: even though not possible - but we just initiate

### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND** 

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [11]:
# Write for loops to iterate over the whole Viterbi Value matrix. 
# Each time, call the function 

In [12]:
# Write a function to trace the state path that gave the maximum probability. 
# This will be the final result. 


# HINT: You should first find the maximum value in the last column of `viterbi_value_matrix`,
# because that is the one with the largest probability. 
# The index of that value is the state of the last nucleotide.  

In [13]:
def _log(p):
    """Safely compute log; returns -inf when p is zero."""
    if p > 0:
        return math.log(p)
    return float('-inf')

n_states  = len(hidden_states)
n_columns = len(observed_seq)

# Allocate the two Viterbi matrices
viterbi_value_matrix = np.full((n_states, n_columns), float('-inf'))
viterbi_trace_matrix = np.zeros((n_states, n_columns), dtype=int)

# --- Seed the first column (observation at t = 0) ---
first_obs = nuc_index[observed_seq[0]]
for st in range(n_states):
    t_prob = trans_matrix[0][st]
    e_prob = emit_matrix[st][first_obs]
    viterbi_value_matrix[st, 0] = _log(t_prob) + _log(e_prob)
    viterbi_trace_matrix[st, 0] = 0  # everything originates from start state

In [14]:
def calculate_prob_for_a_node(target_state, prev_col, obs_idx):
    """Compute the best score arriving at *target_state* given the previous column."""
    best_score = float('-inf')
    best_origin = -1
    for origin in range(len(prev_col)):
        tp = trans_matrix[origin][target_state]
        ep = emit_matrix[target_state][obs_idx]
        score = prev_col[origin] + _log(tp) + _log(ep)
        if score > best_score:
            best_score = score
            best_origin = origin
    return best_score, best_origin

# --- Fill the rest of the trellis column by column ---
col = 1
while col < n_columns:
    obs_idx = nuc_index[observed_seq[col]]
    prev_col = viterbi_value_matrix[:, col - 1]
    for st in range(n_states):
        val, origin = calculate_prob_for_a_node(st, prev_col, obs_idx)
        viterbi_value_matrix[st, col] = val
        viterbi_trace_matrix[st, col] = origin
    col += 1

In [15]:
def traceback_viterbi(val_mat, trace_mat):
    """Walk backwards through the trace matrix to recover the optimal state path."""
    total_cols = val_mat.shape[1]

    # Identify the best-scoring state in the final column
    final_col = val_mat[:, -1]
    terminal_state = int(np.argmax(final_col))
    optimal_log_prob = final_col[terminal_state]

    # Collect states in reverse order
    reversed_path = [terminal_state]
    ptr = terminal_state
    for step in range(total_cols - 1, 0, -1):
        ptr = trace_mat[ptr, step]
        reversed_path.append(ptr)

    # Flip to get chronological order and convert indices to labels
    reversed_path.reverse()
    decoded = "".join(state_labels[s] for s in reversed_path)

    return decoded, optimal_log_prob

# --- Run traceback and display results ---
best_state_sequence, best_prob = traceback_viterbi(viterbi_value_matrix, viterbi_trace_matrix)

print("Query Sequence:   ", observed_seq)
print("Most Likely Path: ", best_state_sequence)
print("Max Log Prob:     ", best_prob)

Query Sequence:    CTTCATGTGAAAGCAGACGTAAGTCA
Most Likely Path:  EEEEEEEEEEEEEEEEEEEEEEEEEE
Max Log Prob:      -38.677666280562796
